<a href="https://colab.research.google.com/github/afon6ka-a11y/A-B-testing/blob/main/Portfolio_project_2_Kurchenko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
from google.colab import files
from google.cloud import bigquery
import numpy as np
import pandas as pd
from scipy import stats

## **1.Exporting a database from BigQuery**
**Вивантаження бази даних з Big Query**

In [ ]:
# Connection to the database in Google BigQuery (DA dataset) / Підключення до бази даних в Google BigQuery (DA dataset)
# Authentication / Аутентифікація
auth.authenticate_user()

In [ ]:
# Creating a client for BigQuery / Створення клієнта для BigQuery
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
# Writing a SQL query in the Python to create a dataset containing all the information required for analysis
#Написання SQL запиту у Python для створення набору даних, що містить усю необхідну для аналізу інформацію

query = """
WITH
 session_info AS (
   SELECT
     s.date,
     s.ga_session_id,
     sp.country,
     sp.device,
     sp.continent,
     sp.channel,
     ab.test,
     ab.test_group
   FROM `data-analytics-mate.DA.ab_test` ab
   JOIN `DA.session` s
     ON ab.ga_session_id = s.ga_session_id
   JOIN `DA.session_params` sp
     ON sp.ga_session_id = ab.ga_session_id
 ),
 session_with_orders AS (
   SELECT
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     COUNT(DISTINCT o.ga_session_id) AS session_with_orders
   FROM `DA.order` o
   JOIN session_info si
     ON o.ga_session_id = si.ga_session_id
   GROUP BY
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group
 ),
 events AS (
   SELECT
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     ep.event_name,
     COUNT(ep.ga_session_id) AS event_cnt
   FROM `DA.event_params` ep
   JOIN session_info si
     ON ep.ga_session_id = si.ga_session_id
   GROUP BY
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     ep.event_name
 ),
 session AS (
   SELECT
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     COUNT(DISTINCT si.ga_session_id) AS session_cnt
   FROM session_info si
   GROUP BY
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group
 ),
 account AS (
   SELECT
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
   FROM `DA.account_session` acs
   JOIN session_info si
     ON acs.ga_session_id = si.ga_session_id
   GROUP BY
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group
 )
SELECT
 swo.date,
 swo.country,
 swo.device,
 swo.continent,
 swo.channel,
 swo.test,
 swo.test_group,
 'session_with_orders' AS event_name,
 swo.session_with_orders AS value
FROM session_with_orders swo
UNION ALL
SELECT
 events.date,
 events.country,
 events.device,
 events.continent,
 events.channel,
 events.test,
 events.test_group,
 events.event_name,
 events.event_cnt AS value
FROM events
UNION ALL
SELECT
 session.date,
 session.country,
 session.device,
 session.continent,
 session.channel,
 session.test,
 session.test_group,
 'session' AS event_name,
 session.session_cnt AS value
FROM session
UNION ALL
SELECT
 account.date,
 account.country,
 account.device,
 account.continent,
 account.channel,
 account.test,
 account.test_group,
 'new_account' AS event_name,
 account.new_account_cnt AS value
FROM account
"""

**Description**

The query builds one aggregated dataset for A/B test analysis in long format (one row = one event type for one segment on one day). It is built from several CTEs, whose results are then stacked with UNION ALL.
CTE:
- session_info takes all sessions that participated in an A/B test. The ab_test  table is joined with session table  (date) and  session_params table (country, device, continent, channel). Each session gets the test name and the group (test, test_group). The other CTEs are built on top of it, so all of them contain only sessions from the experiment.
- session_with_orders counts unique sessions that have at least one order (COUNT(DISTINCT o.ga_session_id)), by date, country, device, continent, channel, test and test group.
- events counts events from event_params by the same segments and additionally by event type (event_name), for example add_to_cart, begin_checkout, add_payment_info.
- session counts unique sessions (session_cnt) in each segment. This is the denominator for conversion metrics.
account counts unique sessions in which a new account was registered (new_account_cnt), via account_session.

In the final SELECT, the four results are combined with UNION ALL. All of them have the same set of columns: date, country, device, continent, channel, test, test_group, event_name and value. Metric names are set as constants: session_with_orders, session, new_account. Event names for the rest come from event_params. As a result, every metric is stored in one value column and distinguished by event_name.

---------
**Опис**

Запит будує один агрегований датасет для аналізу A/B тестів у «довгому» форматі (один рядок = один тип події для одного сегмента за день). Він складається з кількох CTE, результати яких об'єднуються через UNION ALL.
CTE:
- session_info бере всі сесії, що брали участь в A/B тесті. Таблиця ab_test зʼєднується з таблицею session (дата) і таблицею session_params (країна, пристрій, континент, канал). Кожна сесія отримує назву тесту і групу (test, test_group). На цьому CTE побудовані всі інші, тому в них потрапляють лише сесії з експерименту.
- session_with_orders рахує унікальні сесії, у яких є хоча б одне замовлення (COUNT(DISTINCT o.ga_session_id)), у розрізі дати, країни, пристрою, континенту, каналу, тесту й групи тесту.
- events рахує події з event_params за тими самими сегментами та додатково за типом події (event_name), наприклад add_to_cart, begin_checkout, add_payment_info.
- session рахує унікальні сесії (session_cnt) у кожному сегменті. Це знаменник для метрик конверсії.
account рахує унікальні сесії, у яких зареєстровано новий акаунт (new_account_cnt), через account_session.

У фінальному SELECT чотири результати об'єднуються через UNION ALL. Усі вони мають однаковий набір колонок: date, country, device, continent, channel, test, test_group, event_name і value. Назви метрик задані константами: session_with_orders, session, new_account. Назви решти подій беруться з event_params. У результаті всі метрики лежать в одній колонці value і розрізняються за event_name.

In [ ]:
# Processing the SQL query and result / Виконання запиту та результат
query_job = client.query(query)
results = query_job.result()

In [ ]:
# Converting the results to a DataFrame / Перетворення результатів на DataFrame
data = results.to_dataframe()

### **2. A detailed analysis of the contents of dataset and clean data**
**Детальний аналіз змісту датасету та очищення даних**

In [ ]:
# Show info about our dataframe / Виводимо інформацію про датафрейм
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


In [ ]:
# Count of columns by dtype / Підрахунок кількості стовпців за типом даних
print(data.dtypes.value_counts())

object    5
Int64     3
dbdate    1
Name: count, dtype: int64


In [ ]:
# Number of rows and columns / Кількість рядків та стовпців
print(data.shape)

(800996, 9)


Total numbers of rows - 800996 and columns - 9
- 5 columns with object dtype
- 3 columns with int dtype
- 1 column with date dtype will change on datetime


-----------
Загальна кількість рядків - 800996 та колонок - 9
- 5 стовпців з типом даних "object"
- 3 стовпці з типом даних "int"
- 1 стовпець з типом даних "date" замінимо на "datetime"


In [ ]:
# Show fisrt 10 rows / Виводимо перші 10 рядків
data.head(10)

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,(not set),desktop,Americas,Paid Search,2,1,session_with_orders,1
1,2020-11-01,(not set),desktop,Europe,Paid Search,2,2,session_with_orders,1
2,2020-11-02,(not set),mobile,Americas,Direct,2,1,session_with_orders,1
3,2020-11-02,(not set),desktop,Africa,Organic Search,2,2,session_with_orders,1
4,2020-11-02,(not set),mobile,(not set),Paid Search,2,1,session_with_orders,1
5,2020-11-03,(not set),mobile,Asia,Direct,2,1,session_with_orders,1
6,2020-11-03,(not set),desktop,(not set),Paid Search,2,1,session_with_orders,1
7,2020-11-03,(not set),desktop,Asia,Organic Search,2,2,session_with_orders,1
8,2020-11-03,(not set),desktop,Americas,Organic Search,2,2,session_with_orders,1
9,2020-11-03,(not set),desktop,(not set),Organic Search,2,1,session_with_orders,1


In [ ]:
# Show last 10 rows / Виводимо остані 10 рядків
data.tail(10)

,date,country,device,continent,channel,test,test_group,event_name,value
800986,2020-11-29,Vietnam,mobile,Asia,Social Search,2,1,session_start,1
800987,2020-11-29,Vietnam,mobile,Asia,Organic Search,3,2,session_start,1
800988,2020-12-27,Vietnam,mobile,Asia,Paid Search,4,2,first_visit,1
800989,2020-11-04,Vietnam,desktop,Asia,Direct,2,2,scroll,1
800990,2020-11-16,Vietnam,mobile,Asia,Direct,1,2,page_view,1
800991,2020-11-22,Vietnam,mobile,Asia,Social Search,3,1,scroll,1
800992,2020-11-22,Vietnam,mobile,Asia,Social Search,2,1,scroll,1
800993,2020-11-26,Vietnam,mobile,Asia,Paid Search,1,1,first_visit,1
800994,2020-11-26,Vietnam,mobile,Asia,Paid Search,2,1,first_visit,1
800995,2020-12-11,Vietnam,desktop,Asia,Direct,4,1,view_promotion,1


In [ ]:
# Unique values per column / Унікальні значення у кожному стовпці
print(data.nunique().sort_values(ascending=False)) # Sorting by descending / сортування від більшого до меншого

value         936
country       108
date           88
event_name     19
continent       6
channel         5
test            4
device          3
test_group      2
dtype: int64


In [ ]:
# Show information about columns with int values / Виводимо інформацію про колонки з int даними
print(data.describe())

           test  test_group      value
count  800996.0    800996.0   800996.0
mean   2.831191    1.499608   9.475863
std     1.11694         0.5  37.493267
min         1.0         1.0        1.0
25%         2.0         1.0        1.0
50%         3.0         1.0        2.0
75%         4.0         2.0        5.0
max         4.0         2.0     1575.0


In [ ]:
# Number of fully duplicated rows / Кількість повністю дубльованих рядків
print('Number of fully dublicated rows:', data.duplicated().sum())

Number of fully dublicated rows: 0


In [ ]:
# Number of NaN in each column / Кількість пропусків у кожній колонці
print(data.isna().sum())

date          0
country       0
device        0
continent     0
channel       0
test          0
test_group    0
event_name    0
value         0
dtype: int64


In [ ]:
# Information about column 'date' / Інформація про стовпець 'date'
# Period of the data and number of days / Період, за який наведено дані та кількість днів
print(f'Period of data from: {data['date'].min()} till {data['date'].max()}')
print(f'Number of days: {data['date'].nunique()}')


Period of data from: 2020-11-01 till 2021-01-27
Number of days: 88


In [ ]:
# Transform dtype to datetime / Перетворюємо тип даних у datetime
data['date'] = pd.to_datetime(data['date'])

The **date** column contains the date of each session (year, month, day) US format. The period covers 1 November 2020 to 27 January 2021, inclusive. Number of days: 88 days
Сonverted the data in the column to the datetime64[ns] dtype, which is the standard type for working with dates in pandas.

-----------------

Стовпець **date** містить дату кожної сесії (рік, місяць, день) US формат. Період охоплює з 1 листопада 2020 року по 27 січня 2021 року включно. Загальна кількість днів: 88.
Перетворили данні у стовпеці на тип datetime64[ns], який є стандартним для роботи з датами в pandas.

In [ ]:
# Information about column 'country' / Інформація про стовпець 'country'
# Counts values by country / # Підрахунок значень по країнам
print(data['country'].value_counts())

country
United States     53507
India             38123
Canada            35165
United Kingdom    26455
(not set)         22063
                  ...  
Bahamas             601
Honduras            593
Bahrain             589
Paraguay            581
Kosovo              487
Name: count, Length: 108, dtype: int64


In [ ]:
# Number of unique country / Кількість унікальних країн
print('Number of unique countries:', data['country'].nunique())

Number of unique countries: 108


In [ ]:
# Show a list of countries / Виводимо список країн
print('List of countries:', data['country'].unique())

List of countries: ['(not set)' 'Albania' 'Algeria' 'Argentina' 'Armenia' 'Australia'
 'Austria' 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Belarus'
 'Belgium' 'Bolivia' 'Bosnia & Herzegovina' 'Brazil' 'Bulgaria' 'Cambodia'
 'Canada' 'Chile' 'China' 'Colombia' 'Costa Rica' 'Croatia' 'Cyprus'
 'Czechia' 'Denmark' 'Dominican Republic' 'Ecuador' 'Egypt' 'El Salvador'
 'Estonia' 'Finland' 'France' 'Georgia' 'Germany' 'Ghana' 'Greece'
 'Guatemala' 'Honduras' 'Hong Kong' 'Hungary' 'Iceland' 'India'
 'Indonesia' 'Iraq' 'Ireland' 'Israel' 'Italy' 'Jamaica' 'Japan' 'Jordan'
 'Kazakhstan' 'Kenya' 'Kosovo' 'Kuwait' 'Latvia' 'Lebanon' 'Lithuania'
 'Luxembourg' 'Macao' 'Malaysia' 'Malta' 'Mexico' 'Mongolia' 'Morocco'
 'Myanmar (Burma)' 'Nepal' 'Netherlands' 'New Zealand' 'Nigeria'
 'North Macedonia' 'Norway' 'Oman' 'Pakistan' 'Palestine' 'Panama'
 'Paraguay' 'Peru' 'Philippines' 'Poland' 'Portugal' 'Puerto Rico' 'Qatar'
 'Romania' 'Saudi Arabia' 'Serbia' 'Singapore' 'Slovakia' 'Slovenia'
 'Sout

In [ ]:
# Rename country from (not set) to Unknown / Перейменування (not set) на Unknown
data['country'] = data['country'].replace('(not set)', 'Unknown')

The **country** column contains the country of each sessions and has the object data type. It has 108 unique values, from United States (53,507 sessions) to Kosovo (487 sessions).
Rename country from (not set) to Uknown.

---------

Стовпець **country** містить країну, з якої здійснено кожну сесію, і має тип даних object. У ньому 108 унікальних значень: від United States (53,507 сесій) до Kosovo (487 сесій).
Перейменували країну з (not set) на Unknown


In [ ]:
# Info about column 'device' / Iнформація про стовпець 'device'
# Counts values by device / # Підрахунок значень по пристроям
print(data['device'].value_counts())

device
desktop    419152
mobile     335832
tablet      46012
Name: count, dtype: int64


In [ ]:
# Number of unique device / Кількість унікальних пристроїв
print('Number of unique devices:', data['device'].nunique())

Number of unique devices: 3


In [ ]:
# Share of sessions per device, % / Частка сесій за пристроями, %
print((data['device'].value_counts(normalize=True) * 100).round(1))

device
desktop    52.3
mobile     41.9
tablet      5.7
Name: proportion, dtype: float64


The **device** column contains the device type of each event and has 3 unique values: Desktop (419,152 sessions, 52.3%), Mobile (335,832 sessions, 41,9%) and Tablet (46,021 sessions, 5.7%). Desktop and Mobile together account for 94,3% of all sessions, while Tablet has a small share.

------------

Стовпець **device** містить тип пристрою, з якого здійснено кожну подію, і має 3 унікальні значення: Desktop (419,152 sessions, 52.3%), Mobile (335,832 sessions, 41,9%) та Tablet (46,021 sessions, 5.7%). Desktop і Mobile разом дають 94,3% усіх сесій, а частка Tablet мала.

In [ ]:
# Info about column 'continent' / Iнформація про стовпець 'continent'
# Counts values by continent / # Підрахунок значень по континентам
print(data['continent'].value_counts())

continent
Europe       289723
Asia         285747
Americas     171834
Africa        28652
Oceania       19040
(not set)      6000
Name: count, dtype: int64


In [ ]:
# Rename country from (not set) to Unknown / Перейменування (not set) на Unknown
data['continent'] = data['continent'].replace('(not set)', 'Unknown')

In [ ]:
# Share of sessions per continent, % / Частка сесій за континентами, %
print((data['continent'].value_counts(normalize=True) * 100).round(1))

continent
Europe      36.2
Asia        35.7
Americas    21.5
Africa       3.6
Oceania      2.4
Unknown      0.7
Name: proportion, dtype: float64


The **continent** column contains the continent of each event and has the object data type. It has 6 unique values: Americas, Asia, Europe, Oceania, Africa and Unknown.

Three continents (Americas, Asia and Europe) generate about 93.4% of all sessions.
Rename continent from (not set) to Uknown.

-----------

Стовпець **continent** містить континент, з якого здійснено кожну сесію, і має тип даних object. У ньому 6 унікальних значень: Americas, Asia, Europe, Oceania, Africa та Unknown.

Три континенти (Americas, Asia та Europe) разом дають близько 93,4% усіх сесій.
Перейменували континет з (not set) на Unknown


In [ ]:
# Info about column 'channel' / Iнформація про стовпець по каналам трафіку
# Counts values by channel / # Підрахунок значень по каналам трафіку
print(data['channel'].value_counts())

channel
Organic Search    237152
Paid Search       204056
Direct            188716
Social Search      96068
Undefined          75004
Name: count, dtype: int64


In [ ]:
# Number of unique channel / Кількість унікальних каналів трафіку
print('Number of unique channel:', data['channel'].nunique())

Number of unique channel: 5


In [ ]:
# Share of sessions per channel, % / Частка сеансів за каналам трафіку, %
print((data['channel'].value_counts(normalize=True) * 100).round(1))

channel
Organic Search    29.6
Paid Search       25.5
Direct            23.6
Social Search     12.0
Undefined          9.4
Name: proportion, dtype: float64


In [ ]:
# Rename value from 'Undefined' to 'Unknown' / Перейменовуємо значення 'Undefined' на 'Unknown'
data['channel'] = data['channel'].replace('Undefined', 'Unknown')

The **channel** column contains the marketing channel through which the user came to the website in each event and has 5 unique values.
In order to standardise our data, we are renaming ‘Undefined’ to ‘Unknown’.

---------

Стовпець **channel** містить маркетинговий канал, через який користувач потрапив на сайт у кожній сесії, і має 5 унікальних значень.
З метою стандартизації наших даних ми змінюємо назву 'Undefined' на 'Unknown'.

In [ ]:
# Info about column 'test' / Iнформація про стовпець 'test'
# Number of unique test / Кількість унікальних test
print('Number of unique test:', data['test'].nunique())

Number of unique test: 4


In [ ]:
# Rename values and change dtype into str / Перейменовуємо значення та змінюємо тип даних на int
data['test'] = 'Test ' + data['test'].astype(str)

In [ ]:
# Counts values by test / # Підрахунок значень по test
print(data['test'].value_counts())

test
Test 4    302824
Test 3    200757
Test 2    156791
Test 1    140624
Name: count, dtype: int64


The **test** column shows the number of the A/B test to which the sessions relate. A total of 4 tests were carried out.
Change dtype to object and rename values as Test 1

---------

У стовпці **test** вказано номер A/B-тесту, до якого відносяться сесії. Всього було проведено 4 тести.
Змінюємо тип даних на object та перейменовуємо як Test 1

In [ ]:
# Info about column 'test_group' / Iнформація про стовпець 'test_group'
# Number of unique test_group / Кількість унікальних test_group
print('Number of unique test_group:', data['test_group'].nunique())

Number of unique test_group: 2


In [ ]:
# Rename column from 'test_group' to 'test group' / Перейменовуємо стовпець з 'test_group' на 'test group'
data = data.rename(columns={'test_group': 'test group'})

In [ ]:
# Rename value from 1 and 2 to A and B / Перейменовіємо
data['test group'] = data['test group'].map({1: 'A', 2: 'B'})

In [ ]:
# Counts values by test group / # Підрахунок значень по test group
print(data['test group'].value_counts())

test group
A    400812
B    400184
Name: count, dtype: int64


In [ ]:
# Share of sessions per test group, % / Частка сеансів за test group, %
print((data['test group'].value_counts(normalize=True) * 100).round(1))

test group
A    50.0
B    50.0
Name: proportion, dtype: float64


The **test group** column shows which variant of the A/B test a record belongs to: the control group or the test group. We checked it: it has 2 unique values. We renamed it to test group and replaced the numeric values 1 and 2 with the group names A and B. After that we counted the rows per group: A has 400,812 and B has 400,184, which is 50.0% each, so the traffic split matches the planned 50/50.

---------
Колонка **test group** показує, до якого варіанта A/B тесту належить запис: до контрольної чи до тестової групи. Ми перевірили її: у ній 2 унікальні значення. Перейменували її на test group і замінили числові значення 1 і 2 на назви груп A та B. Після цього порахували кількість рядків по групах: A має 400,812, B має 400,184, тобто по 50,0% кожна, отже розподіл трафіку відповідає запланованому 50/50.

In [ ]:
# Info about column 'event_name' / Iнформація про стовпець 'event_name'
# Number of unique event_name / Кількість унікальних event_name
print('Number of unique event_name:', data['event_name'].nunique())

Number of unique event_name: 19


In [ ]:
# Name of unique event_name / Назви унікальних event_name
print('Name of unique event_name:', data['event_name'].unique())

Name of unique event_name: ['session_with_orders' 'new_account' 'session' 'scroll' 'page_view'
 'user_engagement' 'view_item' 'session_start' 'first_visit'
 'begin_checkout' 'view_promotion' 'add_to_cart' 'select_item'
 'select_promotion' 'view_search_results' 'add_shipping_info'
 'add_payment_info' 'click' 'view_item_list']


In [ ]:
# Rename column from 'event_name' to 'event name' / Перейменовуємо стовпець з 'test_group' на 'test group'
data = data.rename(columns={'event_name': 'event name'})

In [ ]:
# Delete '_' from name of events and capitalilse name of event  / Видаляємо '_' з назви event та робимо назву події з великої літери
data['event name'] = data['event name'].str.replace('_', ' ').str.capitalize()

In [ ]:
# Counts values by event name/ # Підрахунок значень по event name
print(data['event name'].value_counts())

event name
Session                107210
Session start          106242
Page view              101907
User engagement         94520
First visit             81621
Scroll                  73643
View promotion          61695
View item               44869
Session with orders     25892
New account             22389
View search results     14310
Add shipping info       11961
Begin checkout          11959
Select item             11816
Add to cart             11762
Select promotion         9044
Add payment info         8048
Click                    1997
View item list            111
Name: count, dtype: int64


The **event_name** column contains the type of user action or metric recorded in the data, for example a session, page view, add to cart or begin checkout. We checked it: it has 19 unique event types. We renamed the column to event name, replaced underscores with spaces and capitalized the first letter of each name (session_with_orders became Session with orders). After that we counted the rows per event: the most frequent are Session (107,210), Session start (106,242) and Page view (101,907).

---------

Колонка **event_name** містить тип дії користувача або метрики, зафіксованої в даних, наприклад сесія, перегляд сторінки, додавання в кошик чи початок оформлення. Ми перевірили її: у ній 19 унікальних типів подій. Перейменували колонку на event name, замінили підкреслення на пробіли й зробили назви з великої літери (session_with_orders стало Session with orders). Після цього порахували кількість рядків по кожній події: найчастіші Session (107,210), Session start (106,242) і Page view (101,907).

In [ ]:
# Info about column 'value' / Iнформація про стовпець 'value'
# Number of unique value / Кількість унікальних value
print('Number of unique value:', data['value'].nunique())

Number of unique value: 936


The **value** column holds the numeric amount for each row: the number of events (or unique sessions) of the type in event name for one day and one segment. Because the data is aggregated and in long format, value must be summed, and conversion is calculated as the sum of value for an event divided by the sum for Session in the same segment and test group.

------------

Колонка **value** містить числове значення для кожного рядка: кількість подій (або унікальних сесій) типу з event name за один день і в одному сегменті. Оскільки дані агреговані й у «довгому» форматі, value потрібно сумувати, а конверсія рахується як сума value потрібної події, поділена на суму value для Session у тому самому сегменті й групі тесту.

In [ ]:
# Cleaned data (10 rows) / Очищенні данні (10 рядків)
data.head(10)

,date,country,device,continent,channel,test,test group,event name,value
0,2020-11-01,Unknown,desktop,Americas,Paid Search,Test 2,A,Session with orders,1
1,2020-11-01,Unknown,desktop,Europe,Paid Search,Test 2,B,Session with orders,1
2,2020-11-02,Unknown,mobile,Americas,Direct,Test 2,A,Session with orders,1
3,2020-11-02,Unknown,desktop,Africa,Organic Search,Test 2,B,Session with orders,1
4,2020-11-02,Unknown,mobile,Unknown,Paid Search,Test 2,A,Session with orders,1
5,2020-11-03,Unknown,mobile,Asia,Direct,Test 2,A,Session with orders,1
6,2020-11-03,Unknown,desktop,Unknown,Paid Search,Test 2,A,Session with orders,1
7,2020-11-03,Unknown,desktop,Asia,Organic Search,Test 2,B,Session with orders,1
8,2020-11-03,Unknown,desktop,Americas,Organic Search,Test 2,B,Session with orders,1
9,2020-11-03,Unknown,desktop,Unknown,Organic Search,Test 2,A,Session with orders,1


In [ ]:
# General info about cleaned data / Загальна інформація про очищені данні
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  object        
 6   test group  800996 non-null  object        
 7   event name  800996 non-null  object        
 8   value       800996 non-null  Int64         
dtypes: Int64(1), datetime64[ns](1), object(7)
memory usage: 55.8+ MB


In [ ]:
#To create an analytical dashboard in Tableau Public. Convert the database to CSV format and download it / Для створення аналітичного дашборду у Tableau Public. Перетворюємо базу даних у csv формат та скачуємо
# Download cleaned dataframe for working with it in Tableau / Скачуємо очищений датафрейм для роботи з ним у Tableau

#data.to_csv('data_cleaned.csv', index=False)
#files.download('data_cleaned.csv')

**Summary: Data description**
* Purpose: aggregated A/B test data, in long format (one metric per row).
* Total number of columns: 9
* Rows: 800,996
* Period of data: from 1 November 2020 to 27 January 2021, inclusive (88 days).
* Data types after conversion: datetime64 (1), int64 (1), object (7). Before conversion: object (5), int64 (3), dbdate (1). The columns test and test group were converted from int to object, date from dbdate to datetime64[ns].
* Categorical columns: 7: country, device, continent, channel, test, test group, event name.
* Datetime columns: 1: date.
* Numeric columns: 1: value (the only column used for calculations).
* Number of A/B tests: 4.
* Test groups: 2 (A and B), 400,812 and 400,184 rows, 50.0% each, so the split matches the planned 50/50.
* Missing values: not found
* Hidden placeholders: Unknown in country, continent and channel (renamed from (not set) and Undefined).
* Number of fully duplicated rows: not found
* Audience breakdown (share of rows): Desktop 52.3%, Mobile 41.9%, Tablet 5.7%. Americas, Asia and Europe give about 93.4%.

---------

**Підсумок: Опис даних**
* Призначення: агреговані дані A/B тестів у «довгому» форматі (одна метрика в одному рядку).
* Загальна кількість стовпців: 9
* Рядки: 800 996
* Період даних: з 1 листопада 2020 до 27 січня 2021 включно (88 днів).
* Типи даних після перетворення: datetime64 (1), int64 (1), object (7). До перетворення: object (5), int64 (3), dbdate (1). Стовпці test і test group перетворено з int на object, date з dbdate на datetime64[ns].
* Категоріальні стовпці: 7: country, device, continent, channel, test, test group, event name.
* Стовпці з датою: 1: date.
* Числові стовпці: 1: value (єдиний стовпець для розрахунків).
* Кількість A/B тестів: 4.
* Груп тесту: 2 (A і B), 400 812 і 400 184 рядки, по 50,0%, тож розподіл відповідає запланованому 50/50.
* Пропущені значення: немає
* Приховані пропуски: Unknown у country, continent і channel (перейменовано з (not set) і Undefined).
* Кількість повністю дубльованих рядків: немає
* Склад аудиторії (частка рядків): Desktop 52,3%, Mobile 41,9%, Tablet 5,7%. Americas, Asia та Europe разом дають близько 93,4%.



### **3. Calculation of statistical significance**
**Розрахунок статистичної значущості**

In [ ]:
# Constants & Configuration Setup / Налаштування констант та конфігурацій
# 1. Configuration of 4 metrics / Конфігурація 4 метрик
METRICS_CONFIG = {
    'Add payment info/Session': {
        'num_event': 'Add payment info', #numerator / чисельник
        'den_event': 'Session', #denominator / знаменик
    },
    'Add shipping info/Session': {
        'num_event': 'Add shipping info',
        'den_event': 'Session',
    },
    'Begin checkout/Session': {
        'num_event': 'Begin checkout',
        'den_event': 'Session',
    },
    'New accounts/Session': {
        'num_event': 'New account',
        'den_event': 'Session',
    },
}

# 2. Data segmentation / Сегменти для розрізу даних
SEGMENTS = ['country', 'device', 'continent', 'channel']

# 3. Configuring column names and group options / Налаштування імен колонок та варіантів груп
TEST_COL = 'test'  # Name of test / Назва тесту ('Test 2')
GROUP_COL = 'test group'  # Group ('A' та 'B')
VARIANT_1 = 'A'  # Control group /Контрольна група
VARIANT_2 = 'B'  # Test group / Тестова група
VALUE_COL = 'value'
EVENT_COL = 'event name'

This block sets up the constants and configuration for the significance calculation. It defines the 4 metrics as a dictionary: for each metric, the numerator event (for example, Add payment info) and the denominator event (Session). It lists the segments used to break down the results: country, device, continent and channel. It also fixes the names of the key columns (test, test group, event name, value) and the labels of the groups: A is the control and B is the test group. Keeping all of this in one place means the later code uses loops over these settings, so a new metric or segment can be added without changing the calculation logic.

--------------

Цей блок задає константи та конфігурацію для розрахунку статистичної значущості. Він описує 4 метрики у вигляді словника: для кожної вказано подію в чисельнику (наприклад, Add payment info) і подію в знаменнику (Session). Також задано список сегментів для розрізів: country, device, continent і channel. Окремо зафіксовано назви ключових колонок (test, test group, event name, value) і позначення груп: A це контрольна, B тестова. Усі налаштування зібрано в одному місці, тому подальший код працює в циклах за цими значеннями, а нову метрику чи сегмент можна додати без зміни логіки розрахунку.

In [ ]:
# Z-Test Calculation Function / Функція для розрахунку Z-тесту
# Calculates the Z-statistic, p-value, relative change in conversion rate and significance
# Обчислює Z-статистику, p-value, відносну зміну конверсії та значущість
def calculate_z_test(count1, nobs1, count2, nobs2, alpha=0.05):
  if nobs1 == 0 or nobs2 == 0:
    return {
        'conv_1': 0.0,
        'conv_2': 0.0,
        'change_pct': 0.0,
        'z_stat': 0.0,
        'p_value': 1.0,
        'significant': False,
    }

  conv1 = count1 / nobs1
  conv2 = count2 / nobs2

  # Percentage change / Відносна зміна у %
  change_pct = ((conv2 - conv1) / conv1) * 100 if conv1 > 0 else 0.0

  # Pooled proportion constrained to the range [0, 1] to guard against repeated events
  # Pooled proportion з обмеженням в межах [0, 1] для захисту від подій із повтореннями
  p_pooled = (count1 + count2) / (nobs1 + nobs2)
  p_pooled = np.clip(p_pooled, 0.0, 1.0)

  # Standard error / Стандартна похибка
  se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / nobs1 + 1 / nobs2))

  if se == 0:
    z_stat = 0.0
    p_val = 1.0
  else:
    z_stat = (conv2 - conv1) / se
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))

  return {
      'conv_1': conv1,
      'conv_2': conv2,
      'change_pct': change_pct,
      'z_stat': z_stat,
      'p_value': p_val,
      'significant': p_val < alpha,
  }

This block defines the function calculate_z_test, which runs a two-sided z-test for two proportions (group A vs group B). It takes the number of events and the number of sessions in each group and returns a dictionary with the conversion in A and B, the relative change in %, the z-statistic, the p-value and a significance flag (p-value below alpha = 0.05). If one group has no sessions, the function returns neutral values (p-value = 1, not significant) instead of an error. The same happens when the standard error is 0. The pooled proportion is limited to the range from 0 to 1, because for events that can repeat within a session the number of events can exceed the number of sessions. The function is called in a loop for every test, metric and segment.

----------------

Цей блок задає функцію calculate_z_test, яка виконує двосторонній z-тест для двох пропорцій (група A проти групи B). Вона приймає кількість подій і кількість сесій у кожній групі та повертає словник з конверсією в A і B, відносною зміною в %, z-статистикою, p-value і ознакою значущості (p-value менше за alpha = 0,05). Якщо в одній із груп немає сесій, функція повертає нейтральні значення (p-value = 1, незначуще) замість помилки. Те саме відбувається, коли стандартна похибка дорівнює 0. Загальна частка (pooled proportion) обмежується діапазоном від 0 до 1, бо для подій, що можуть повторюватися в сесії, кількість подій може перевищувати кількість сесій. Функція викликається в циклі для кожного тесту, метрики та сегмента.

In [ ]:
# Processing Total & Segmented A/B Test Results / Обробка результатів (Total + Сегментація)
results = []

# Getting a list of all unique tests / Отримуємо список усіх унікальних тестів
tests = data[TEST_COL].unique()

for test_name in tests:
  data_test = data[data[TEST_COL] == test_name]

  # Total Calculation (Without segmentation) / Розрахунок Total (Без сегментації)
  for metric_name, config in METRICS_CONFIG.items():
    num_evt = config['num_event']
    den_evt = config['den_event']

    # Group A / Група А
    g1 = data_test[data_test[GROUP_COL] == VARIANT_1]
    n1 = g1[g1[EVENT_COL] == num_evt][VALUE_COL].sum()
    d1 = g1[g1[EVENT_COL] == den_evt][VALUE_COL].sum()

    # Group B / Група В
    g2 = data_test[data_test[GROUP_COL] == VARIANT_2]
    n2 = g2[g2[EVENT_COL] == num_evt][VALUE_COL].sum()
    d2 = g2[g2[EVENT_COL] == den_evt][VALUE_COL].sum()

    stats_res = calculate_z_test(n1, d1, n2, d2)

    results.append({
        'test_number': test_name,
        'segment': 'Total',
        'segment_value': 'Total',
        'metric': metric_name,
        'numerator_event': num_evt,
        'denominator_event': den_evt,
        'numerator_count_1': n1,
        'denominator_count_1': d1,
        'conversion_1': stats_res['conv_1'],
        'numerator_count_2': n2,
        'denominator_count_2': d2,
        'conversion_2': stats_res['conv_2'],
        'metric_change_%': stats_res['change_pct'],
        'z_stat': stats_res['z_stat'],
        'p_value': stats_res['p_value'],
        'significant': stats_res['significant'],
    })

  #  Calculation by segments / Розрахунок у розрізі сегментів
  for seg in SEGMENTS:
    if seg not in data_test.columns:
      continue

    unique_vals = data_test[seg].dropna().unique()

    for val in unique_vals:
      data_seg = data_test[data_test[seg] == val]

      for metric_name, config in METRICS_CONFIG.items():
        num_evt = config['num_event']
        den_evt = config['den_event']

        g1 = data_seg[data_seg[GROUP_COL] == VARIANT_1]
        n1 = g1[g1[EVENT_COL] == num_evt][VALUE_COL].sum()
        d1 = g1[g1[EVENT_COL] == den_evt][VALUE_COL].sum()

        g2 = data_seg[data_seg[GROUP_COL] == VARIANT_2]
        n2 = g2[g2[EVENT_COL] == num_evt][VALUE_COL].sum()
        d2 = g2[g2[EVENT_COL] == den_evt][VALUE_COL].sum()

        if d1 == 0 and d2 == 0:
          continue

        stats_res = calculate_z_test(n1, d1, n2, d2)

        results.append({
            'test_number': test_name,
            'metric': metric_name,
            'numerator_event': num_evt,
            'denominator_event': den_evt,
            'numerator_count_1': n1,
            'denominator_count_1': d1,
            'conversion_1': stats_res['conv_1'],
            'numerator_count_2': n2,
            'denominator_count_2': d2,
            'conversion_2': stats_res['conv_2'],
            'metric_change_%': stats_res['change_pct'],
            'z_stat': stats_res['z_stat'],
            'p_value': stats_res['p_value'],
            'significant': stats_res['significant'],
            'segment': seg,
            'segment_value': val,
        })

# Creating the final DataFrame / Формування фінального DataFrame
final_df = pd.DataFrame(results)


This block calculates the significance results for all tests, metrics and segments and collects them in one table. It loops over every unique test and, for each one, first computes the Total level (no segmentation) and then the segment level (country, device, continent, channel). For every metric, it takes the sum of value for the numerator event and for the denominator event (Session) separately in group A and group B, and passes these four numbers to calculate_z_test. Each result is saved as one row with the test, segment, metric, counts, conversions, change in %, z-statistic, p-value and significance flag. At the segment level, combinations where both groups have no sessions are skipped. In the end, the list of rows is turned into the DataFrame final_df, which is exported and used in the Tableau dashboard.

-------

Цей блок розраховує результати значущості для всіх тестів, метрик і сегментів та збирає їх в одну таблицю. Він проходить циклом по кожному унікальному тесту й для кожного спочатку рахує рівень Total (без сегментації), а потім рівень сегментів (country, device, continent, channel). Для кожної метрики окремо в групах A і B береться сума value для події в чисельнику й для події в знаменнику (Session), і ці чотири числа передаються у calculate_z_test. Кожен результат зберігається як один рядок: тест, сегмент, метрика, кількості, конверсії, зміна в %, z-статистика, p-value та ознака значущості. На рівні сегментів комбінації, де в обох групах немає сесій, пропускаються. Наприкінці список рядків перетворюється на DataFrame final_df, який експортується й використовується в дашборді Tableau.

In [ ]:
# Checking shape of dataframe / Перевірка розмірності датафрейму
print(f'Number of columns: {final_df.shape[1]}')
print(f'Number of rows: {final_df.shape[0]}')

# Show first 10 rows / Виводимо перші 10 рядків
final_df.head(10)

Number of columns: 16
Number of rows: 1968


,test_number,segment,segment_value,metric,numerator_event,denominator_event,numerator_count_1,denominator_count_1,conversion_1,numerator_count_2,denominator_count_2,conversion_2,metric_change_%,z_stat,p_value,significant
0,Test 2,Total,Total,Add payment info/Session,Add payment info,Session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
1,Test 2,Total,Total,Add shipping info/Session,Add shipping info,Session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
2,Test 2,Total,Total,Begin checkout/Session,Begin checkout,Session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
3,Test 2,Total,Total,New accounts/Session,New account,Session,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
4,Test 2,country,Unknown,Add payment info/Session,Add payment info,Session,23,384,0.059896,18,421,0.042755,-28.617164,-1.104814,0.269240,False
5,Test 2,country,Unknown,Add shipping info/Session,Add shipping info,Session,28,384,0.072917,26,421,0.061758,-15.303699,-0.632137,0.527298,False
6,Test 2,country,Unknown,Begin checkout/Session,Begin checkout,Session,32,384,0.083333,35,421,0.083135,-0.237530,-0.010155,0.991898,False
7,Test 2,country,Unknown,New accounts/Session,New account,Session,27,384,0.070312,37,421,0.087886,24.993402,0.920584,0.357268,False
8,Test 2,country,Albania,Add payment info/Session,Add payment info,Session,0,14,0.000000,1,14,0.071429,0.000000,1.018350,0.308512,False
9,Test 2,country,Albania,Add shipping info/Session,Add shipping info,Session,1,14,0.071429,2,14,0.142857,100.000000,0.611010,0.541193,False


In [ ]:
# Final total results for 4th tests / Фінальні загальні результати для 4х тестів
total_df = final_df[final_df['segment'] == 'Total'].reset_index(drop=True)
print(total_df.shape)
total_df

(16, 16)


,test_number,segment,segment_value,metric,numerator_event,denominator_event,numerator_count_1,denominator_count_1,conversion_1,numerator_count_2,denominator_count_2,conversion_2,metric_change_%,z_stat,p_value,significant
0,Test 2,Total,Total,Add payment info/Session,Add payment info,Session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
1,Test 2,Total,Total,Add shipping info/Session,Add shipping info,Session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
2,Test 2,Total,Total,Begin checkout/Session,Begin checkout,Session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
3,Test 2,Total,Total,New accounts/Session,New account,Session,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
4,Test 1,Total,Total,Add payment info/Session,Add payment info,Session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
5,Test 1,Total,Total,Add shipping info/Session,Add shipping info,Session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
6,Test 1,Total,Total,Begin checkout/Session,Begin checkout,Session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
7,Test 1,Total,Total,New accounts/Session,New account,Session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
8,Test 4,Total,Total,Add payment info/Session,Add payment info,Session,3731,105079,0.035507,3601,105141,0.034249,-3.541234,-1.571106,0.116158,False
9,Test 4,Total,Total,Add shipping info/Session,Add shipping info,Session,5128,105079,0.048801,4956,105141,0.047137,-3.411125,-1.785795,0.074132,False


**Final conclusion**

The analysis covers 4  A/B tests (Test 1 to Test 4) on the website. The data covers 1 November 2020 to 27 January 2021 (88 days). The initial dataset has 800,996 rows and 9 columns. It is aggregated and in long format: one row is one event type for one day and one segment. In total, the tests include 542,142 sessions (Test 1: 90,555, Test 2: 100,881, Test 3: 140,486, Test 4: 210,220).
The tests are analyzed separately, but they were not fully separate in time. Tests 1-3 overlap on 19-26 November, and Tests 3-4 overlap on 5-20 December. In these periods, the same users could take part in more than one test, so the effect of one test may mix with another.
The traffic split is even. Groups A and B have 400,812 and 400,184 sessions (50.0% each). The session counts in each test are also very close, for example 50,637 and 50,244 in Test 2, so no problem with the split was found.

**Analysis:**

Four metrics were checked, each as events per session: add_payment_info, add_shipping_info, begin_checkout and new_account. They were calculated for each test in total, and also by country, continent, device and channel. A two-sided z-test for two proportions was used, with a significance level of 0.05. This test fits because each metric is a rate (events divided by sessions) and the two groups are independent.
The results table has 16 rows at the total level (4 tests × 4 metrics). It contains the conversion in A and B, the change in %, the z-statistic, the p-value and the significance flag. The table final_df is used in the Tableau dashboard (1968 rows, columns 16). Decisions are based on the total level, and the segments help to look for the reasons behind the results.

**Results by test**
* Test 1: implement B. Three metrics grew significantly: add_payment_info +12.54% (p < 0.001), add_shipping_info +6.56% (p = 0.009) and begin_checkout +6.66% (p = 0.003). New accounts fell by 3.35%, but this is not significant (p = 0.123). The change improves the checkout flow. It is recommended to roll it out and keep watching new accounts.
* Test 2: no effect. All four metrics grew a little (+1.2% to +3.6%), but none is significant (p from 0.21 to 0.56). B is not better than A, so rollout is not recommended. More data can be collected, or a stronger change can be tested.
* Test 3: do not implement. Begin_checkout fell by 3.35% (p = 0.012), which is significant. The other metrics show no significant change (p from 0.16 to 0.52). B made the result worse.
* Test 4: do not implement. Begin_checkout fell by 2.35% (p = 0.046) and new accounts fell by 3.36% (p = 0.018). Add_payment_info and add_shipping_info also went down, but not significantly (p = 0.116 and 0.074). All four metrics are lower in B.

---------------

**Фінальний висновок**

Аналіз охоплює 4 A/B тести (Test 1-Test 4) на сайті. Дані за період з 1 листопада 2020 до 27 січня 2021 (88 днів). Початковий датасет містить 800 996 рядків і 9 колонок. Він агрегований і в «довгому» форматі: один рядок це один тип події за один день в одному сегменті. Разом у тестах 542 142 сесії (Test 1: 90 555, Test 2: 100 881, Test 3: 140 486, Test 4: 210 220).
Тести аналізуються окремо, але в часі вони не були повністю розділені. Test 1-3 перетинаються 19-26 листопада, а Test 3-4 перетинаються 5-20 грудня. У ці періоди одні й ті самі користувачі могли брати участь у кількох тестах, тому ефект одного тесту може змішуватися з іншим.
Розподіл трафіку рівномірний. У групах A і B 400 812 та 400 184 сесії (по 50,0%). Кількість сесій у кожному тесті теж дуже близька, наприклад 50 637 і 50 244 у Test 2, тому проблем із розподілом не виявлено.

**Аналіз**

Перевірялися чотири метрики, кожна як кількість подій на сесію: add_payment_info, add_shipping_info, begin_checkout і new_account. Їх розраховано для кожного тесту в цілому, а також у розрізі країн, континентів, пристроїв і каналів. Використано двосторонній z-тест для двох пропорцій з рівнем значущості 0,05. Він підходить, бо кожна метрика це частка (події до сесій), а групи незалежні.
Таблиця результатів на рівні total має 16 рядків (4 тести × 4 метрики). У ній конверсія в A і B, зміна в %, z-статистика, p-value і ознака значущості. Таблиця final_df використовується в дашборді Tableau (1968 рядків, 16 колонок). Рішення ухвалюються за рівнем total, а сегменти допомагають шукати причини результатів.

**Результати по тестах:**
* Test 1: впроваджувати B. Три метрики значуще зросли: add_payment_info +12,54% (p < 0,001), add_shipping_info +6,56% (p = 0,009) і begin_checkout +6,66% (p = 0,003). Нові акаунти знизилися на 3,35%, але це незначуще (p = 0,123). Зміна покращує шлях до оплати. Рекомендовано впровадити її й далі стежити за новими акаунтами.
* Test 2: ефекту немає. Усі чотири метрики трохи зросли (від +1,2% до +3,6%), але жодна не значуща (p від 0,21 до 0,56). B не кращий за A, тому впровадження не рекомендоване. Можна зібрати більше даних або протестувати сильнішу зміну.
* Test 3: не впроваджувати. Begin_checkout знизився на 3,35% (p = 0,012), це значуще. В інших метриках значущих змін немає (p від 0,16 до 0,52). B погіршив результат.
* Test 4: не впроваджувати. Begin_checkout знизився на 2,35% (p = 0,046), нові акаунти на 3,36% (p = 0,018). Add_payment_info і add_shipping_info також нижчі, але незначуще (p = 0,116 і 0,074). У B усі чотири метрики нижчі.


In [ ]:
#To create an analytical dashboard in Tableau Public. Convert the database to CSV format and download it / Для створення аналітичного дашборду у Tableau Public. Перетворюємо базу даних у csv формат та скачуємо
# Download final and total dataframes for working with it in Tableau / Скачуємо фінальний та загальний датафрейми для роботи з ним у Tableau

#final_df.to_csv('final_df.csv', index=False)
#files.download('final_df.csv')

#total_df.to_csv('total_df.csv', index=False)
#files.download('total_df.csv')

# link for csv file
# final:
# https://drive.google.com/file/d/1fI0jfxEwEqWlhvzfixJc8uuBF5P_rO4e/view?usp=sharing

# total
# https://drive.google.com/file/d/1ScvBNXlDFH-p2unDIU7D7GwiWXRkGbVz/view?usp=sharing


### **4. Using the data obtained in Google Colab, we created a dashboard in Tableau Public: ‘A/B Testing Analytics Dashboard’ / На основі даних, отриманих у Google Colab, ми створили Дашборд у Tableau Public: «Інформаційна панель аналітики A/B-тестування»**

https://public.tableau.com/views/ABTestingAnalyticsDashboard/ABTestingAnalyticsDashboard?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
